In [ ]:
import json
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

# --- 1️⃣ Load CySecBERT for embeddings ---
cybert_model_name = "markusbayer/CySecBERT"
cybert_tokenizer = AutoTokenizer.from_pretrained(cybert_model_name)
cybert_model = AutoModel.from_pretrained(cybert_model_name)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cybert_model.to(device)

def get_embedding(text):
    inputs = cybert_tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = cybert_model(**inputs)
    emb = outputs.last_hidden_state.mean(dim=1)
    return emb / emb.norm(dim=-1, keepdim=True)

# --- 2️⃣ Threat dataset (sample entries) ---
threat_dataset = [
    {
        "domain": "Web",
        "component": "Web Server",
        "threat_type": "Tampering",
        "attack_scenario": "SQL injection in login form",
        "risk_level": "High",
        "mitigation": "Parameterized queries, input validation",
        "security_baseline_reference": "MSB-SEC-02",
        "questions": ["Are inputs sanitized?", "Is there a WAF?"],
        "red_flags": ["Direct SQL queries", "No input validation"],
        "similarity_score": None
    },
    {
        "domain": "Web",
        "component": "Frontend",
        "threat_type": "Tampering",
        "attack_scenario": "Cross-site scripting in comments",
        "risk_level": "Medium",
        "mitigation": "Input sanitization, CSP headers",
        "security_baseline_reference": "MSB-SEC-06",
        "questions": ["Are inputs sanitized?", "Is CSP enforced?"],
        "red_flags": ["No input encoding", "Inline scripts allowed"],
        "similarity_score": None
    },
    {
        "domain": "Mobile",
        "component": "Android App",
        "threat_type": "Elevation of Privilege",
        "attack_scenario": "App exploits system vulnerability to gain root access",
        "risk_level": "High",
        "mitigation": "Apply OS patches, sandboxing, secure coding",
        "security_baseline_reference": "MSB-SEC-04",
        "questions": ["Is OS updated?", "Is app sandboxed?"],
        "red_flags": ["Outdated SDK", "Improper permission handling"],
        "similarity_score": None
    },
]

# Embed dataset
dataset_embeddings = torch.vstack([get_embedding(t["attack_scenario"]) for t in threat_dataset]).cpu()

# --- 3️⃣ Load generative model (Mistral/LLaMA) ---
gen_model_name = "mistralai/Mistral-7B-v0.1"  # or LLaMA variant
gen_tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
gen_model = AutoModelForCausalLM.from_pretrained(gen_model_name, torch_dtype=torch.float16, device_map="auto")
gen_model.eval()

# --- 4️⃣ RAG + JSON structured generation ---
def generate_structured_threats(query, top_k=2, num_generate=3):
    # Clear CUDA cache before generation to free up memory
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # Retrieve top-k relevant threats
    query_emb = get_embedding(query).cpu()
    scores = torch.nn.functional.cosine_similarity(query_emb, dataset_embeddings)
    topk_indices = scores.topk(top_k).indices.tolist()
    retrieved = [threat_dataset[i] for i in topk_indices]

    # Build JSON schema prompt
    context_json = json.dumps(retrieved, indent=4)
    prompt = f"""
You are a cybersecurity threat modeling assistant.
Given the project description: "{query}".
Use these example threats as context (JSON format):

{context_json}

Generate {num_generate} detailed threat entries in **exactly the same JSON schema**:
domain, component, threat_type, attack_scenario, risk_level, mitigation,
security_baseline_reference, questions, red_flags, similarity_score (set None for now).
Return only JSON.
"""
    inputs = gen_tokenizer(prompt, return_tensors="pt").to(device)
    # Reduced max_new_tokens to mitigate CUDA out of memory error
    outputs = gen_model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.7)
    generated_text = gen_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Parse JSON output
    try:
        # The model tends to repeat the prompt in its output. We need to extract only the JSON part.
        # Attempt to find the first '[' and last ']' to isolate the JSON array.
        json_start_index = generated_text.find('[')
        json_end_index = generated_text.rfind(']')
        if json_start_index != -1 and json_end_index != -1 and json_end_index > json_start_index:
            clean_json_string = generated_text[json_start_index : json_end_index + 1]
            structured_output = json.loads(clean_json_string)
        else:
            # If JSON structure is not clearly found, fallback to attempt direct parse or wrap
            structured_output = json.loads(generated_text)

    except json.JSONDecodeError:
        # fallback: wrap in array or handle as error
        print(f"JSON parsing failed for output: {generated_text}")
        # Try to clean common unwanted text before JSON
        potential_json = generated_text.split('Return only JSON')[-1].strip()
        try:
            structured_output = json.loads(potential_json)
        except json.JSONDecodeError:
            structured_output = [{"error": "Could not parse JSON", "raw_output": generated_text}]

    return structured_output

#Display as table ---
def display_threats_table(threats):
    df = pd.DataFrame(threats)
    # Expand lists for questions & red_flags into strings
    df["questions"] = df["questions"].apply(lambda x: ", ".join(x) if isinstance(x, list) else str(x))
    df["red_flags"] = df["red_flags"].apply(lambda x: ", ".join(x) if isinstance(x, list) else str(x))
    return df

#Example usage ---
domain = "Web"
components = "React, Angular, PWA"
query = f"Domain: {domain}. Components: {components}"

generated_threats = generate_structured_threats(query)
df = display_threats_table(generated_threats)
print(df)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: markusbayer/CySecBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [ ]:
import json
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM

# -----------------------------
# 1️⃣ Load CySecBERT (Embeddings)
# -----------------------------
cybert_model_name = "markusbayer/CySecBERT"

cybert_tokenizer = AutoTokenizer.from_pretrained(cybert_model_name)
cybert_model = AutoModel.from_pretrained(cybert_model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cybert_model.to(device)

def get_embedding(text):
    inputs = cybert_tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = cybert_model(**inputs)

    emb = outputs.last_hidden_state.mean(dim=1)
    emb = emb / emb.norm(dim=-1, keepdim=True)

    return emb.cpu()

# -----------------------------
# 2️⃣ Threat Dataset (Anchors)
# -----------------------------
threat_dataset = [
    {
        "domain": "Web",
        "component": "Web Server",
        "threat_type": "Tampering",
        "attack_scenario": "SQL injection vulnerability in login form",
        "risk_level": "High",
        "mitigation": "Use parameterized queries and input validation",
        "security_baseline_reference": "MSB-SEC-02",
        "questions": ["Are inputs sanitized?", "Is there a WAF?"],
        "red_flags": ["Direct SQL queries", "No input validation"],
        "similarity_score": None
    },
    {
        "domain": "Web",
        "component": "Frontend",
        "threat_type": "Tampering",
        "attack_scenario": "Cross site scripting in comment section",
        "risk_level": "Medium",
        "mitigation": "Input sanitization and Content Security Policy",
        "security_baseline_reference": "MSB-SEC-06",
        "questions": ["Is CSP enabled?", "Are inputs escaped?"],
        "red_flags": ["Inline scripts", "No input filtering"],
        "similarity_score": None
    },
    {
        "domain": "Mobile",
        "component": "Android App",
        "threat_type": "Elevation of Privilege",
        "attack_scenario": "Application exploits OS vulnerability to gain root access",
        "risk_level": "High",
        "mitigation": "Update SDKs and enforce sandboxing",
        "security_baseline_reference": "MSB-SEC-04",
        "questions": ["Are SDKs updated?", "Are permissions restricted?"],
        "red_flags": ["Outdated SDK", "Excessive permissions"],
        "similarity_score": None
    }
]

# -----------------------------
#Embed Dataset
dataset_embeddings = torch.vstack(
    [get_embedding(t["attack_scenario"]) for t in threat_dataset]
)

# -----------------------------
#Load Generator (FLAN-T5)
gen_model_name = "google/flan-t5-base"  #use base (faster + stable)

gen_tokenizer = AutoTokenizer.from_pretrained(gen_model_name)
gen_model = AutoModelForSeq2SeqLM.from_pretrained(gen_model_name).to(device)
gen_model.eval()

# -----------------------------
#RAG + Generation
def generate_structured_threats(query, top_k=2, num_generate=5):

    # --- Retrieve ---
    query_emb = get_embedding(query)

    scores = torch.nn.functional.cosine_similarity(
        query_emb,
        dataset_embeddings
    )

    topk_indices = scores.topk(top_k).indices.tolist()
    retrieved = [threat_dataset[i] for i in topk_indices]

    context_json = json.dumps(retrieved, indent=2)

    # --- Prompt (STRICT + REASONING) ---
    prompt = f"""
You are a cybersecurity threat modeling expert.

Project:
{query}

Context examples:
{context_json}

TASK:
Generate {num_generate} NEW cybersecurity threats using reasoning.

STRICT RULES:
- Output MUST be valid JSON
- Output MUST be a LIST
- Do NOT explain anything
- Do NOT add text outside JSON

FORMAT:
[
{{
"domain": "Web",
"component": "Example",
"threat_type": "Tampering",
"attack_scenario": "Detailed realistic attack",
"risk_level": "High",
"mitigation": "Specific mitigation",
"security_baseline_reference": "MSB-XXX",
"questions": ["Q1", "Q2"],
"red_flags": ["R1", "R2"],
"similarity_score": null
}}
]
"""

    inputs = gen_tokenizer(prompt, return_tensors="pt").to(device)

    # --- Generate (stable settings) ---
    outputs = gen_model.generate(
        **inputs,
        max_new_tokens=400,
        do_sample=False,   # removes randomness
        temperature=0.0    # forces consistency
    )

    text = gen_tokenizer.decode(outputs[0], skip_special_tokens=True)

    # --- Clean JSON ---
    try:
        start = text.find("[")
        end = text.rfind("]") + 1
        clean_json = text[start:end]

        return json.loads(clean_json)

    except:
        print("JSON parsing failed — using fallback dataset")
        return retrieved

# -----------------------------
#Display Table
def display_threats_table(threats):

    df = pd.DataFrame(threats)

    for col in ["questions", "red_flags"]:
        if col in df.columns:
            df[col] = df[col].apply(
                lambda x: ", ".join(x) if isinstance(x, list) else str(x)
            )

    return df

# -----------------------------
#Example Run
domain = "Web"
components = "React, Angular, PWA"
query = f"Domain: {domain}. Components: {components}"
generated = generate_structured_threats(query)
df = display_threats_table(generated)
print(df)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/664 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/321 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (555 > 512). Running this sequence through the model will result in indexing errors
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


JSON parsing failed — using fallback dataset
   domain    component             threat_type  \
0  Mobile  Android App  Elevation of Privilege   
1     Web     Frontend               Tampering   

                                     attack_scenario risk_level  \
0  Application exploits OS vulnerability to gain ...       High   
1            Cross site scripting in comment section     Medium   

                                       mitigation security_baseline_reference  \
0              Update SDKs and enforce sandboxing                  MSB-SEC-04   
1  Input sanitization and Content Security Policy                  MSB-SEC-06   

                                        questions  \
0  Are SDKs updated?, Are permissions restricted?   
1            Is CSP enabled?, Are inputs escaped?   

                             red_flags similarity_score  
0  Outdated SDK, Excessive permissions             None  
1   Inline scripts, No input filtering             None  
